In [4]:
import requests
import csv
import time
import random
session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0 (compatible; bg3-reviews-scraper/1.0)"})

APP_ID = 1086940
MAX_REVIEWS = 100000
REVIEWS_PER_PAGE = 100  # Steam API limit
LANGUAGE = "english"
FILTER = "recent"

In [5]:
url = f"https://store.steampowered.com/appreviews/{APP_ID}"
params = {
    "json": 1,
    "filter": FILTER,
    "language": LANGUAGE,
    "num_per_page": REVIEWS_PER_PAGE,
    "cursor": "*"
}

In [ ]:
all_reviews = []
review_count = 0
SAVE_EVERY = 1000
MAX_ATTEMPTS = 5
while review_count < MAX_REVIEWS:
    attempt = 0
    backoff = 1
    success = False
    data = None
    while attempt < MAX_ATTEMPTS and not success:
        try:
            resp = session.get(url, params=params, timeout=30)
            if resp.status_code == 200:
                try:
                    data = resp.json()
                except ValueError:
                    print("JSON decode error — retrying...")
                    attempt += 1
                    time.sleep(backoff)
                    backoff *= 2
                    continue
                success = True
            elif resp.status_code in (502, 503, 504):
                attempt += 1
                print(f"Transient HTTP {resp.status_code} —" 
                      f"retry {attempt}/{MAX_ATTEMPTS} after {backoff}s"
                      )
                time.sleep(backoff)
                backoff *= 2
            else:
                print("HTTP error:", resp.status_code)
                success = False
                break
        except requests.exceptions.RequestException as e:
            attempt += 1
            print(f"Request exception: {e} — retrying in {backoff}s")
            time.sleep(backoff)
            backoff *= 2
    if not success:
        print("Error - brake")
        break

    reviews = data.get("reviews", [])
    if not reviews:
        print("Brak kolejnych recenzji")
        break

    for review in reviews:
        all_reviews.append({
            "review_id": review.get("recommendationid"),
            "review": review.get("review"),
            "voted_up": review.get("voted_up"),
            "timestamp_created": review.get("timestamp_created"),
            "author_steamid": review.get("author", {}).get("steamid")
        })

    review_count += len(reviews)
    print(f"Pobrano {review_count} recenzji...")
    params["cursor"] = data.get("cursor", params.get("cursor"))

    # zapis częściowy co 1000 recenzji
    if review_count % SAVE_EVERY == 0:
        with open("english_bg3_reviews_partial.csv", mode="w", newline="", encoding="utf-8") as file:
            writer = csv.DictWriter(file, fieldnames=all_reviews[0].keys(), delimiter=';')
            writer.writeheader()
            writer.writerows(all_reviews)
        print(f"Zapisano cząstkowo {len(all_reviews)} recenzji.")

    time.sleep(1 + random.random())

Pobrano 100 recenzji...
Pobrano 200 recenzji...
Pobrano 300 recenzji...
Pobrano 400 recenzji...
Pobrano 500 recenzji...
Pobrano 600 recenzji...
Pobrano 700 recenzji...
Pobrano 800 recenzji...
Pobrano 900 recenzji...
Pobrano 1000 recenzji...
Zapisano cząstkowo 1000 recenzji.
Pobrano 1100 recenzji...
Pobrano 1200 recenzji...
Pobrano 1300 recenzji...
Pobrano 1400 recenzji...
Pobrano 1500 recenzji...
Pobrano 1600 recenzji...
Pobrano 1700 recenzji...
Pobrano 1800 recenzji...
Pobrano 1900 recenzji...
Pobrano 2000 recenzji...
Zapisano cząstkowo 2000 recenzji.
Pobrano 2100 recenzji...
Pobrano 2200 recenzji...
Pobrano 2300 recenzji...
Pobrano 2400 recenzji...
Pobrano 2500 recenzji...
Pobrano 2600 recenzji...
Pobrano 2700 recenzji...
Pobrano 2800 recenzji...
Pobrano 2900 recenzji...
Pobrano 3000 recenzji...
Zapisano cząstkowo 3000 recenzji.
Pobrano 3100 recenzji...
Pobrano 3200 recenzji...
Pobrano 3300 recenzji...
Pobrano 3400 recenzji...
Pobrano 3500 recenzji...
Pobrano 3600 recenzji...
Pobrano

In [7]:
with open("english_bg3_reviews.csv", mode="w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=all_reviews[0].keys(),delimiter=';')
    writer.writeheader()
    writer.writerows(all_reviews)
    
print(f"Zapisano {len(all_reviews)} recenzji do 'english_bg3_reviews.csv'.")

Zapisano 100098 recenzji do 'english_bg3_reviews.csv'.
